# What's new in tsauditor 0.4.0

Four changes since 0.3.0, each with its own section below:

1. **PRF007** — infinite values (`inf` / `-inf`) are now detected and repaired
2. **PNL004** — panel rows with a null entity id are now reported instead of
   silently receiving zero checks
3. **`fix()` accepts `available_at=` and `constraints=`** — LEK004 and
   VAL001/VAL002 can now run as part of a one-shot repair
4. **LEK002's default threshold changed** — this is a **behaviour change**:
   some columns flagged under 0.3.0 will no longer be flagged

If you're new to tsauditor entirely, start with
[`examples/getting_started/`](../getting_started) instead — this notebook
assumes you already know `scan()` / `apply_fixes()` / `fix()`.

Install or upgrade: `pip install -U tsauditor`

---

## 1. PRF007 — infinite values

Before 0.4.0, `inf` and `-inf` were invisible to the whole library.
`isna()` is `False` for an infinity, so the missing-data checks never saw one,
and every anomaly/leakage detector quietly replaced it with `NaN` on its own
working copy so its arithmetic wouldn't break. You could run `scan()`, see
nothing relevant, run `fix()`, and still hand infinities to your model.

A feature built from a ratio is the realistic way this happens — a
denominator of zero on some rows is common in ordinary feature engineering,
not a data-entry mistake.

In [1]:
import numpy as np
import pandas as pd
import tsauditor as tsa

idx = pd.bdate_range("2024-01-02", periods=100)
rng = np.random.default_rng(0)

price = pd.Series(100 + np.cumsum(rng.normal(0, 1, 100)), index=idx)
change_pct = price.pct_change()

# A ratio feature whose denominator hits exactly zero a few times -- this is
# how infinities actually show up in real feature pipelines, not by accident.
volume = rng.integers(1000, 5000, 100).astype(float)
volume[[10, 40, 70]] = 0.0
turnover_ratio = price / volume  # divide-by-zero -> +inf on those 3 rows

df = pd.DataFrame(
    {
        "price": price,
        "change_pct": change_pct,
        "turnover_ratio": turnover_ratio,
    }
)
print("inf count in turnover_ratio:", np.isinf(df["turnover_ratio"]).sum())

inf count in turnover_ratio: 3


In [2]:
report = tsa.scan(df, run_stationarity=False)
pd.DataFrame(report.filter(code="PRF007")[0].to_dict(), index=[0])[
    ["code", "severity", "column", "description"]
]

,code,severity,column,description
0,PRF007,critical,turnover_ratio,Column 'turnover_ratio' contains 3 infinite va...


The evidence tells you exactly how much of the column is still usable, and
whether that's enough for the leakage checks to trust it:

In [3]:
report.filter(code="PRF007")[0].evidence

{'non_finite_count': 3,
 'positive_inf_count': 3,
 'negative_inf_count': 0,
 'non_finite_percentage': 3.0,
 'n_finite_remaining': 97,
 'below_leakage_min_obs': False,
 'leakage_min_obs': 30,
 'first_occurrence': '2024-01-16 00:00:00'}

`below_leakage_min_obs` is the field to actually read — below 30 finite
observations, `turnover_ratio` wouldn't just be noisier for LEK001/LEK002/
LEK003/LEK005, it would be **skipped by them entirely**, with no message.

`apply_fixes()` converts the infinities to `NaN` and imputes them, and — unlike
every other repair — this step is unconditional. There is no reading under
which keeping an infinity is correct, so it isn't gated by `outliers=` or
`missing=` the way clipping and interpolation are.

In [4]:
clean = report.apply_fixes(df)
print("inf before:", np.isinf(df["turnover_ratio"]).sum())
print("inf after :", np.isinf(clean["turnover_ratio"]).sum())
print([f for f in report.last_fixes if f["column"] == "turnover_ratio"])

inf before: 3
inf after : 0
[{'column': 'turnover_ratio', 'action': 'clip_outliers', 'cells_changed': 0, 'bounds': (nan, nan)}, {'column': 'turnover_ratio', 'action': 'clip_spikes', 'cells_changed': 3}, {'column': 'turnover_ratio', 'action': 'non_finite_to_nan', 'cells_changed': 3}, {'column': 'turnover_ratio', 'action': 'impute_interpolate', 'cells_changed': 3}]


/usr/local/lib/python3.10/dist-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


---

## 2. PNL004 — rows with a null entity id

`scan(..., group_col=...)` partitions a panel by entity and audits each one as
its own time series. `df.groupby(group_col)` drops rows with a null entity id
by default — the only sound choice, since there's no entity identity to
compare coverage or short-history against.

But before 0.4.0 that meant those rows received **zero** checks and nothing
said so: not the panel-level checks, not any per-entity check either, since
the same `groupby` drives the per-entity loop. A null-id row looked identical
to a clean one. PNL004 reports the count and percentage up front.

In [5]:
rng = np.random.default_rng(1)
dates = pd.date_range("2024-01-01", periods=100, freq="D")
parts = [
    pd.DataFrame(
        {"ticker": t, "price": 100 + np.cumsum(rng.normal(0, 1, 100))},
        index=dates,
    )
    for t in ("AAA", "BBB")
]
panel = pd.concat(parts).sort_index()
panel.iloc[0:20, panel.columns.get_loc("ticker")] = None  # 20 rows, no entity id

panel_report = tsa.scan(panel, group_col="ticker", run_stationarity=False)
pnl004 = panel_report.filter(code="PNL004")[0]
print(pnl004.description)
print()
print(pnl004.evidence)

20 of 200 rows have a null value in the entity column 'ticker'. These rows cannot be assigned to any entity, so they are excluded from every panel check and every per-entity check (leakage, anomaly, profiler) — not flagged clean, simply never examined. They are also left unmodified by apply_fixes(), since there is no single entity's distribution to repair them from.

{'n_null_rows': 20, 'n_total_rows': 200, 'pct_null': 10.0, 'group_col': 'ticker'}


These rows are still left unrepaired by `apply_fixes()` — there is no single
entity's distribution to repair them from — but that skip is now explicit and
logged, instead of an accidental side effect of `NaN != NaN`:

In [6]:
panel_clean = panel_report.apply_fixes(panel)
print([f for f in panel_report.last_fixes if f["action"] == "skip_null_group_rows"])

[{'column': 'ticker', 'action': 'skip_null_group_rows', 'cells_changed': 20}]


---

## 3. `fix()` now accepts `available_at=` and `constraints=`

LEK004 (as-of leakage) and VAL001/VAL002 (validity) are opt-in on `scan()`
because tsauditor can't infer a release schedule or a validity bound on its
own. Before 0.4.0, `fix()` had no parameter to pass either one through, so a
one-shot repair silently skipped them — which reads as "nothing wrong" rather
than "not checked." The only way to exercise them together with a repair was
`scan()` followed by `apply_fixes()` in two separate calls.

In [7]:
cpi = pd.Series(50.0, index=idx)  # a macro series used as a feature
spread = price * 0.001  # bid/ask spread, must stay >= 0
spread.iloc[10:15] = -0.5  # a real violation: negative spread

df2 = pd.DataFrame({"price": price, "cpi": cpi, "spread": spread})

clean2, report2 = tsa.fix(
    df2,
    available_at={"cpi": pd.Timedelta(days=30)},  # cpi is published ~30d late
    constraints={"bounds": {"spread": {"min": 0}}},
)
print("codes found:", sorted({i.code for i in report2.all_issues}))

codes found: ['ANO001', 'ANO002', 'LEK004', 'PRF001', 'PRF003', 'VAL001']


Both `LEK004` and `VAL001` are reachable now in a single call — before 0.4.0,
`fix()` alone would never have produced either code, regardless of what was
passed in. `spread` was deliberately given 5 negative rows above so `VAL001`
has something real to flag rather than staying silent because nothing violated
the bound.

---

## 4. LEK002's default threshold changed — read this if you're upgrading

`audit_correlation_leakage`'s `min_correlation` default moved from **0.1 to
0.5**. This is not a bugfix in the sense of the other three; it's a deliberate,
**silent behaviour change** that will make LEK002 quieter on existing code.

Why: LEK002 fires when a feature's peak cross-correlation with the target lands
at a *positive* lag. For two persistent series (a price level, a random walk,
any slow-moving process), spurious correlation is large by construction while
*which* lag happens to win the argmax is close to a coin flip. Measured over
100 trials per cell on 400-point series, two **independently generated** series
(a true negative by construction) were flagged 37-51% of the time under the old
0.1 gate. Under 0.5, that drops to 8-13%, with no genuine leak lost in 200
trials.

In [8]:
from tsauditor.leakage import audit_correlation_leakage

# Two independently generated random walks -- no real relationship by
# construction, so any finding here is a false positive. seed=2 is used
# because it happens to land in the 0.1-0.5 band this section is about;
# most seeds don't, which is itself the point (see the measured false-positive
# rates below).
rng2 = np.random.default_rng(2)
idx3 = pd.bdate_range("2023-01-02", periods=400)
independent_a = pd.Series(np.cumsum(rng2.normal(0, 1, 400)), index=idx3)
independent_b = pd.Series(np.cumsum(rng2.normal(0, 1, 400)), index=idx3)
df3 = pd.DataFrame({"target": independent_a, "other_series": independent_b})

old_default = audit_correlation_leakage(df3, target="target", min_correlation=0.1)
new_default = audit_correlation_leakage(df3, target="target", min_correlation=0.5)
print(
    "flagged under old default (0.1):",
    [(i.column, i.evidence["peak_correlation"]) for i in old_default],
)
print(
    "flagged under new default (0.5):",
    [(i.column, i.evidence["peak_correlation"]) for i in new_default],
)

flagged under old default (0.1): [('other_series', 0.4971)]
flagged under new default (0.5): []


The old 0.1 default flagged `other_series` as leaking, a false positive: these
two series were generated completely independently, there is nothing for
either to have leaked. The new 0.5 default correctly stays quiet. Whether any
*particular* pair of independent random walks trips the old gate depends on
the random seed, which is exactly the point: at 0.1 it was close to a coin
flip (37-51% false-positive rate measured across seeds, see below), not a real
threshold.

**If you need the old behaviour** (e.g. you've tuned a downstream pipeline
around it, or you'd rather over-flag than under-flag for now), pass
`min_correlation=0.1` explicitly — the parameter didn't go away, only the
default did:

```python
report = tsa.scan(df, target='y', run_leakage=True)   # uses the new 0.5 default

# to restore 0.3.0 behaviour for this one check, call it directly:
from tsauditor.leakage import audit_correlation_leakage
old_style_issues = audit_correlation_leakage(df, target='y', min_correlation=0.1)
```

This affects `domain="finance"` scans most, since random-walk-like price series
are exactly the case the old gate handled worst.

---

## Where to go next

- [`examples/getting_started/`](../getting_started) — the from-zero walkthrough,
  if any of the API above is unfamiliar
- [Issue Code Reference](https://github.com/imann128/tsauditor/wiki/Issue-code-reference) —
  every code, including PRF007 and PNL004, with full evidence tables
- [Panel Data](https://github.com/imann128/tsauditor/wiki/Panel-Data) — PNL001-PNL004
  in full, including the other three panel-only checks
- [CHANGELOG.md](https://github.com/imann128/tsauditor/blob/main/CHANGELOG.md) —
  every change in this release, with the full measurement behind the LEK002
  threshold change above
- Confused about any of this, or need a tutorial for something not covered
  here? [Open a GitHub Discussion](https://github.com/imann128/tsauditor/discussions) —
  a comprehensive tutorial section is actively being built, and questions
  directly shape what gets written next.